# TỔNG HỢP FEATURE ENGINEERING CẦN LÀM GÌ ?

# 0. Bức tranh tổng thể

Dataset hiện tại mỗi dòng là 1 đội trong 1 trận, và các hỉ số là kết quả sau khi trận đã diễn ra. Nhưng mục tiêu của ta là dự đoán trước trận. Vì vậy Feature Engineering phải biến dữ liệu quá khứ thành thông tin phản ánh 'đội này đang chơi thế nào tính đến thời điểm hiện tại'


# 1. Bước 1 - Loại bỏ Data Leakage
- `PLUS_MINUS`: loại hoàn toàn
- `WL`, `WIN`: Label
- `PTS, REB, AST, FGM, FGA...`: Không dùng trực tiếp, nhưng dùng gián tiếp qua rolling average
- `MIN`: tổng phút thi đấu của trận đó, loại

Các cột an toàn để dùng trực tiếp (biết trước trận): `IS_HOME`, `REST_DAYS`, `IS_B2B`, `CURRENT_WIN_PCT`, `WIN_STREAK`, `GAMES_PLAYED_SEASON`

# 2. Rolling Average (Trung bình trượt)

- Ta không biết PTS của trận sắp tới, nhưng ta biết đội đó ghi trung bình bao nhiêu điểm trong 5 - 10 trận gần nhất. Con số này phản ánh phong độ hiện tại - đội đang chơi tốt hay tệ

- Với mỗi đội, tại mỗi thời điểm, tính trung bình N trận đã diễn ra trước đó (không bao gồm trận hiện tại):
```
Ví dụ rolling 5 trận cho PTS của đội ATL:
Trận 1: PTS=113 → rolling = NaN (chưa đủ 5 trận)
Trận 2: PTS=95  → rolling = NaN
Trận 3: PTS=122 → rolling = NaN
Trận 4: PTS=102 → rolling = NaN
Trận 5: PTS=111 → rolling = NaN
Trận 6: PTS=108 → rolling = mean(113,95,122,102,111) = 108.6
Trận 7: PTS=99  → rolling = mean(95,122,102,111,108) = 107.6
```
- Vậy nếu các trận đầu mùa không có dữ liệu của N trận trước thi sao ?
    - Ta sẽ dùng expanding mean: Dùng tất cả dữ liệu đã có, bao nhiêu cũng được. Đầu mùa ít trận thì tính trung bình ít trận, khi đủ 5 trận thì chuyển sang rolling 5
    ```
    Trận 1: PTS=113 → feature = NaN      (chưa có trận nào trước đó)
    Trận 2: PTS=95  → feature = 113.0    (mean 1 trận: [113])
    Trận 3: PTS=122 → feature = 104.0    (mean 2 trận: [113,95])
    Trận 4: PTS=102 → feature = 110.0    (mean 3 trận: [113,95,122])
    Trận 5: PTS=111 → feature = 108.0    (mean 4 trận: [113,95,122,102])
    Trận 6: PTS=108 → feature = 108.6    (rolling 5: [113,95,122,102,111]) ✓
    Trận 7: PTS=99  → feature = 107.6    (rolling 5: [95,122,102,111,108]) ✓
    ```

    - Trong thực tế, nếu dùng Cold Start (nếu chưa đủ N trận thì bỏ luôn) là không hợp lý, khi muốn dự đoán trận thứ 3 của mùa giair, ta không thể nói không đủ dữ liệu nên không dự đoán được. Ta vẫn dự đoán trên dữ liệu 2 trận đã có

- Sử dụng rolling average window = 5 để tính phong độ gần đây của mỗi đội. Đối với các trận đầu mùa giải chưa đủ 5 trận lịch sử, áp dụng expanding mean (trung bình tích lũy) từ tất cả trận đã có. Phương pháp này vừa tối đa hóa lượng dữ liệu sử dụng, vừa phản ánh đúng tình huống dự đoán thực tế

# 3. Ghép 2 đội thành 1 dòng

- Hiện tại mỗi trận có 2 dòng (đội nhà + đội khách). Nhưng khi dự đoán, input phải là 1 dòng cho 1 trận chứa thông tin của cả 2 đội. Model cần nhìn thấy cả 2 bên để so sánh


In [5]:
import pandas as pd

df = pd.read_csv(r'C:\Code_AI\CS114-FinalTerm\nba_data\nba_full_dataset.csv')

# Lấy 1 trận cụ thể làm ví dụ
sample_game = df[df['GAME_ID'] == df['GAME_ID'].iloc[0]]

print('=' * 70)
print(f'VÍ DỤ: GAME_ID = {sample_game['GAME_ID'].iloc[0]}')
print(f'Ngày: {sample_game['GAME_DATE'].iloc[0]}')
print('=' * 70)

cols = ['GAME_ID', 'TEAM_ABBREVIATION', 'MATCHUP', 'IS_HOME', 'WL',
'PTS', 'REB', 'AST', 'FG_PCT', 'TOV', 'CURRENT_WIN_PCT', 'WIN_STREAK', 'REST_DAYS']

print('\n Hiện tại: 1 trận = 2 dòng')
print('-' * 70)
print(sample_game[cols].to_string(index=False))

print('\n\n' + '=' * 70)
print('Sau khi ghép')
home = sample_game[sample_game['IS_HOME'] == 1].iloc[0]
away = sample_game[sample_game['IS_HOME'] == 0].iloc[0]

print(f"""
GAME_ID:          {home['GAME_ID']}
GAME_DATE:        {home['GAME_DATE']}
HOME_TEAM:        {home['TEAM_ABBREVIATION']}
AWAY_TEAM:        {away['TEAM_ABBREVIATION']}
---
HOME_PTS:         {home['PTS']}          AWAY_PTS:         {away['PTS']}
HOME_REB:         {home['REB']}            AWAY_REB:         {away['REB']}
HOME_AST:         {home['AST']}            AWAY_AST:         {away['AST']}
HOME_FG_PCT:      {home['FG_PCT']}        AWAY_FG_PCT:      {away['FG_PCT']}
HOME_TOV:         {home['TOV']}            AWAY_TOV:         {away['TOV']}
HOME_WIN_PCT:     {home['CURRENT_WIN_PCT']:.3f}       AWAY_WIN_PCT:     {away['CURRENT_WIN_PCT']:.3f}
HOME_WIN_STREAK:  {home['WIN_STREAK']}             AWAY_WIN_STREAK:  {away['WIN_STREAK']}
HOME_REST_DAYS:   {home['REST_DAYS']}           AWAY_REST_DAYS:   {away['REST_DAYS']}
---
LABEL (HOME_WIN): {home['WL']} → {1 if home['WL']=='W' else 0}
""")


# Đếm số dòng
print("=" * 70)
print("TỔNG KẾT:")
print(f"  Hiện tại:  {len(df)} dòng ({df['GAME_ID'].nunique()} trận × 2 dòng/trận)")
print(f"  Sau ghép:  {df['GAME_ID'].nunique()} dòng (1 dòng/trận)")
print("=" * 70)

VÍ DỤ: GAME_ID = 22100014
Ngày: 2021-10-21

 Hiện tại: 1 trận = 2 dòng
----------------------------------------------------------------------
 GAME_ID TEAM_ABBREVIATION     MATCHUP  IS_HOME WL  PTS  REB  AST  FG_PCT  TOV  CURRENT_WIN_PCT  WIN_STREAK  REST_DAYS
22100014               ATL ATL vs. DAL        1  W  113   55   31   0.479   13              0.5           0        7.0
22100014               DAL   DAL @ ATL        0  L   87   50   16   0.333   15              0.5           0        7.0


Sau khi ghép

GAME_ID:          22100014
GAME_DATE:        2021-10-21
HOME_TEAM:        ATL
AWAY_TEAM:        DAL
---
HOME_PTS:         113          AWAY_PTS:         87
HOME_REB:         55            AWAY_REB:         50
HOME_AST:         31            AWAY_AST:         16
HOME_FG_PCT:      0.479        AWAY_FG_PCT:      0.333
HOME_TOV:         13            AWAY_TOV:         15
HOME_WIN_PCT:     0.500       AWAY_WIN_PCT:     0.500
HOME_WIN_STREAK:  0             AWAY_WIN_STREAK:  0
HOME_REST

# 4. Difference Features

- Model sẽ dễ học hơn nếu tạo sẵn feature so sánh giữa 2 đội, thay vì bắt model tự tính
- Ví dụ: Nếu HOME_rolling_PTS = 115 và AWAY_rolling_PTS = 108, thì chênh lệch = +7 -> đội nhà đang ghi điểm tốt hơn, feature này trực tiếp hơn so với 2 feature riêng lẻ
- Nên tạo difference cho: `rolling_PTS`, `rolling_FG_PCT`, `rolling_AST`, `rolling_TOV`, `CURRENT_WIN_PCT`, `WIN_STREAK`, `REST_DAYS`

# 5. Cap outliers

- Như đã phát hiện trong EDA, `REST_DAYS` có outlier, giá trị này gây nhiueex vì thực tế nghỉ 10 ngày hay 150 ngày dều mang ý nghĩa như nhau: đội đã nghỉ đủ
- Chỉ cần `df['REST_DAYS] = df['REST_DAYS].clip(upper=10)`

# 6. Train/Test split theo thời gian

- Đây là bài toán time-series, KHÔNG ĐƯỢC dùng random split. Nếu random split, model có thể train trên trận tháng 3/2025 và test trên trận tháng 1/2025 → model "nhìn thấy tương lai" → kết quả ảo.
- Cách chia: Dùng các mùa 2021-22 đến 2024-25 làm train, mùa 2025-26 làm test. Hoặc chia theo ngày cụ thể (ví dụ: trước 1/10/2025 = train, sau = test). Cách này phản ánh đúng thực tế: ta dùng dữ liệu quá khứ để dự đoán trận chưa diễn ra.

# 7. CÁCH THỰC HIỆN

- Ta sẽ chia ra 2 bộ features khi code:
    - Bộ nhỏ (~ 15 features): chỉ dùng difference features + IS_HOME + REST_DAYS
    - Bộ đầy đủ (~ 40 features): HOME + AWAY riêng + difference. 

- Feature Selection
    - Sau khi tạo xong tất cả features, dùng các kỹ thuật chọn lọc `feature_importance` từ Random Forest / XGBoost cho biết feature nào model thực sự dùng, loại bỏ các feature có importance gần 0. 
    - Regularization: Logistic Regeression có tham số `C` (regularization), XGBoost có `max_depth`, `reg_alpha`, `reg_lambda`. Các tham số này tự động phạt model nếu dùng quá nhiều features -> giảm overfitting
    - Ta sẽ chia ra 2 bộ features khi code:
        - Bộ nhỏ (~ 15 features): chỉ dùng difference features + IS_HOME + REST_DAYS
        - Bộ đầy đủ (~ 40 features): HOME + AWAY riêng + difference. 